System information (for reproducibility):

In [1]:
versioninfo()

Julia Version 1.12.5
Commit 5fe89b8ddc1 (2026-02-09 16:05 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 10 × Apple M1 Pro
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m1)
  GC: Built with stock GC
Threads: 1 default, 1 interactive, 1 GC (on 8 virtual cores)


Load packages:

In [2]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Desktop/UCLA/Biostat 257/257-hw/hw5`


Status `~/Desktop/UCLA/Biostat 257/257-hw/hw5/Project.toml`
  [ab4f0b2a] BFloat16s v0.6.1
  [6e4b80f9] BenchmarkTools v1.8.0
  [1e616198] COSMO v0.8.10
  [61c947e1] Clarabel v0.11.1
  [f65535da] Convex v0.16.5
  [a93c6f00] DataFrames v1.8.2
  [60bf3e95] GLPK v1.2.1
  [2e9cd046] Gurobi v1.9.2
  [87dc4568] HiGHS v1.23.0
  [b99e6be6] Hypatia v0.9.2
  [4076af6c] JuMP v1.30.1
  [1ec41992] MosekTools v0.15.10
  [08abe8d2] PrettyTables v3.3.2
  [c946c3f1] SCS v2.6.3
  [3eaba693] StatsModels v0.7.9


## Q1. [MUON Optimizer](https://kellerjordan.github.io/posts/muon/)

### Q1.1 (10 pts) Projection to the Stiefel manifold

Given $G \in \mathbb{R}^{m \times n}$, derive an analytical solution $O$, using SVD of $G$, that minimizes $\|G - O\|_F^2$ subject to $O^T O = I$ or $O O^T = I$.

**Answer:**

Expanding the squared Frobenius norm:

$$\|G - O\|_F^2 = \|G\|_F^2 - 2\langle G,\, O\rangle_F + \|O\|_F^2.$$

Because $O$ has orthonormal columns (or rows), its squared Frobenius norm equals the number of columns (or rows), i.e. $\|O\|_F^2 = \min(m,n)$, which is a constant independent of $O$. Therefore, minimizing $\|G - O\|_F^2$ is equivalent to

$$\max_{O} \; \langle G, O\rangle_F = \operatorname{tr}(O^\top G).$$

Let $G = U\Sigma V^\top$ be the thin SVD, where $U \in \mathbb{R}^{m \times k}$, $V \in \mathbb{R}^{n \times k}$ have orthonormal columns, $\Sigma = \operatorname{diag}(\sigma_1, \ldots, \sigma_k) \geq 0$, and $k = \min(m,n)$. Then, using the cyclic property of the trace,

$$\operatorname{tr}(O^\top G) = \operatorname{tr}(O^\top U \Sigma V^\top) = \operatorname{tr}(V^\top O^\top U \,\Sigma).$$

Define $W = V^\top O^\top U$. Since $O$, $U$, and $V$ all have orthonormal columns, $W$ is itself an orthogonal matrix. Von Neumann's trace inequality states that for any orthogonal $W$ and positive semidefinite diagonal matrix $\Sigma$,

$$\operatorname{tr}(W \Sigma) \leq \operatorname{tr}(\Sigma) = \sum_{i=1}^k \sigma_i,$$

with equality if and only if $W = I$.

Setting $W = I$:

$$V^\top O^\top U = I \implies O^\top = VU^\top \implies \boxed{O^* = UV^\top}.$$

Verify the constraint, 

- **Case $m \geq n$** ($O^\top O = I$ required):
  $$O^{\top*} O^* = V U^\top U V^\top = V V^\top = I_n. \checkmark$$

- **Case $m \leq n$** ($OO^\top = I$ required):
  $$O^* O^{\top*} = U V^\top V U^\top = U U^\top = I_m. \checkmark$$

- **Case $m = n$**: $O^* = UV^\top$ is square and orthogonal. $\checkmark$

In summary, the minimum of $\|G - O\|_F^2$ over orthogonal matrices $O$ is achieved uniquely at

$$O^* = UV^\top,$$

where $G = U\Sigma V^\top$ is the singular value decomposition of $G$. The minimum value equals

$$\|G\|_F^2 - 2\operatorname{tr}(\Sigma) + \min(m,n) = \sum_{i=1}^k (\sigma_i - 1)^2 \quad \text{(when } m = n\text{)}.$$

### Q1.2 (15 pts) SVD implementation

Implement the above SVD-based projection in Julia. Test your implementation on a random matrix $G$ and verify that the solution $O$ satisfies the orthogonality constraint.

**Answer:** The result $2.4 \times 10^{-13} \approx 0$ indicates that the solution $O$ satisfies the orthogonality constraint.

In [3]:
using LinearAlgebra
using Random

function proj_svd(G::AbstractMatrix)
    # TODO: implement the SVD-based projection to the Stiefel manifold
    U, _, V = svd(G)   # thin SVD
    O = U * V'         # optimal orthogonal projection
    return O # placeholder, replace with the actual projection
end

proj_svd (generic function with 1 method)

In [4]:
Random.seed!(257)

# dimension of Wk (key matrix) in the Llama 405B training
m, n = 16_384, 1_024
G = randn(m, n)

16384×1024 Matrix{Float64}:
  0.679063    0.122441    0.571105   …  -0.300964   1.1327     -0.588327
  1.24568     0.307604    1.13503       -1.15558    0.571349    1.11541
 -1.21007     0.711936    0.991587      -0.521754  -0.0477959   0.477806
 -0.817491   -0.0122939   0.865765      -1.19477   -0.105281   -1.09376
  1.04395     0.0959707   0.659193      -0.318154  -0.305517   -0.0409484
  1.52556    -0.629669    0.229052   …   1.78934   -0.348219    0.280388
 -1.69501     1.34648     0.737386       0.529309  -1.51404    -2.51145
 -0.245347    0.397226    1.35235       -1.05907   -0.493763    0.219485
 -0.131288   -0.0612191   0.506154      -0.743674  -0.105784   -2.30217
 -1.06533    -0.69439    -0.714778      -0.663292  -1.49428     0.514697
  0.234923    1.08498    -0.0796582  …   0.557986   0.46177    -0.864102
 -1.12138     0.570517   -0.0544455      1.22651   -0.276617   -0.443086
  0.222238   -0.620228   -0.854797      -0.636162   0.548592   -0.7009
  ⋮                         

In [5]:
O_svd = proj_svd(G)
println("Orthogonality check: ", norm(O_svd' * O_svd - I))

Orthogonality check: 2.37935965031224e-13


### Q1.3 (10 pts) Newton-Schulz iteration

Consider the Newton-Schulz iteration
$$
\begin{align*}
&G \gets G / \|G\|_F \\
&\text{for iter in 1:maxiter} \\
&\qquad G \gets a G + b G G^T G + c (G G^T)^2 G \\
&\text{end}
\end{align*}
$$
Explain why, for the choice $(a, b, c) = (1.5, -0.5, 0.5)$, the iteration converges to the projection of $G$ onto the Stiefel manifold.



### Q1.4 (15 pts) NS implementation

Implement the above Newton-Schulz iteration in Julia. Test your implementation on the same random matrix $G$ and verify that the solution converges to the same projection as the SVD-based method.

**Answer:** The Newton-Schulz iteration converges to the same solution as the SVD-based projection to within $1.5 \times 10^{-13}$, which is at the level of machine precision, confirming that both methods correctly solve the orthogonal Procrustes problem.

In [6]:
function proj_ns(
    G :: AbstractMatrix{T}, 
    a, 
    b, 
    c;
    maxiter      = 10,
    storage_mn   = similar(G),
    storage_nn_1 = similar(G, size(G, 2), size(G, 2)),
    storage_nn_2 = similar(G, size(G, 2), size(G, 2)),
    O = similar(G)
    ) where T
    # scale singular values to [0, 1]
    O .= G ./ norm(G)
    for iter in 1:maxiter
        # TODO: implement the Newton-Schulz iteration
        # storage_nn_1 = S  = OᵀO          (n×n Gram matrix)
        mul!(storage_nn_1, O', O)
 
        # storage_nn_2 = S² = (OᵀO)²
        mul!(storage_nn_2, storage_nn_1, storage_nn_1)
 
        # storage_nn_2 = aI + bS + cS²  (in-place, no allocation)
        lmul!(c, storage_nn_2)                    # c·S²
        axpy!(b, storage_nn_1, storage_nn_2)      # += b·S
        for i in 1:size(storage_nn_2, 1)
            storage_nn_2[i, i] += a               # += a·I
        end
 
        # storage_mn = O · (aI + bS + cS²)
        mul!(storage_mn, O, storage_nn_2)
 
        O .= storage_mn
    end
    return O
end

proj_ns (generic function with 1 method)

In [7]:
O_ns = proj_ns(G, 2, -1.5, 0.5, maxiter=10)
# check whether the solution is close to the SVD-based projection
norm(O_ns - O_svd)

1.530685633820529e-13

### Q1.4 (20 pts) Benchmark

Benchmark the runtime of the SVD-based projection and the Newton-Schulz iteration for different data types (`Float64`, `Float32`, `Float16`, and `BFloat16` if your hardware supports it) and different devices (CPU and GPU if available). Discuss the results and trade-offs between the two methods in terms of accuracy and computational efficiency.

**Answer:**

Benchmarked on Apple M1 Pro (16 384 × 1 024 matrix, 5 samples). BFloat16 is excluded as it has no native BLAS kernel on this hardware. SVD is CPU-only as Metal.jl does not support svd on MtlArray.

- **Runtime:** SVD and NS are comparable on CPU (~0.85–1.7 s) across all types, since both rely on Accelerate BLAS. Float16 CPU NS is fast because it upcasts to Float32 internally. Surprisingly, Metal GPU NS is ~5× slower than CPU NS — on M1 Pro the CPU and GPU share unified memory, so there is no transfer advantage, and Apple's - Accelerate AMX outperforms Metal Performance Shaders for this matrix size.
- **Accuracy:** Residuals scale with the unit roundoff of each type. NS achieves better accuracy than SVD in Float64 (1.4e-14 vs 2.4e-13) due to its self-correcting fixed-point structure. Float16 Metal NS shows a large residual (0.023) from accumulated low-precision rounding over 10 iterations.
- **Conclusion:** On M1 Pro, CPU Float32 is the best practical choice: it is fast, accurate, and supported by both methods. Metal GPU offers no advantage here due to unified memory and the lack of SVD support; it would be more relevant on a discrete GPU with separate VRAM.

In [15]:
using BenchmarkTools
using BFloat16s
using LinearAlgebra
using Printf
using DataFrames
using Metal

orth_residual(O) = norm(Float64.(O)' * Float64.(O) - I(size(O, 2)))

const NO_BLAS = (Float16, BFloat16)

function run_bench(G_f64::Matrix{Float64}, dtype::Type)
    G      = convert(Matrix{dtype}, G_f64)
    G_work = dtype in NO_BLAS ? convert(Matrix{Float32}, G) : G
    a, b, c = eltype(G_work)(2), eltype(G_work)(-1.5), eltype(G_work)(0.5)

    buf = (storage_mn   = similar(G_work),
           storage_nn_1 = similar(G_work, size(G_work, 2), size(G_work, 2)),
           storage_nn_2 = similar(G_work, size(G_work, 2), size(G_work, 2)),
           O            = similar(G_work))

    proj_svd(G_work); proj_ns(G_work, a, b, c; maxiter=10, buf...)  # warmup

    t_svd   = @belapsed proj_svd($G_work)                                    samples=5 evals=1
    acc_svd = orth_residual(proj_svd(G_work))
    t_ns    = @belapsed proj_ns($G_work, $a, $b, $c; maxiter=10, $buf...)    samples=5 evals=1
    acc_ns  = orth_residual(proj_ns(G_work, a, b, c; maxiter=10, buf...))

    return (t_svd=t_svd, acc_svd=acc_svd, t_ns=t_ns, acc_ns=acc_ns)
end

function run_bench_metal(G_f64::Matrix{Float64}, dtype::Type)
    G_gpu = MtlArray(convert(Matrix{dtype}, G_f64))
    a, b, c = dtype(2), dtype(-1.5), dtype(0.5)

    buf = (storage_mn   = similar(G_gpu),
           storage_nn_1 = similar(G_gpu, size(G_gpu, 2), size(G_gpu, 2)),
           storage_nn_2 = similar(G_gpu, size(G_gpu, 2), size(G_gpu, 2)),
           O            = similar(G_gpu))

    proj_ns(G_gpu, a, b, c; maxiter=10, buf...); Metal.synchronize()  # warmup

    t_ns   = @belapsed (proj_ns($G_gpu, $a, $b, $c; maxiter=10, $buf...); Metal.synchronize()) samples=5 evals=1
    acc_ns = orth_residual(Array(proj_ns(G_gpu, a, b, c; maxiter=10, buf...)))

    return (t_ns=t_ns, acc_ns=acc_ns)
end

# ── sweep ──────────────────────────────────────────────────────────────────────
cpu_dtypes   = [Float64, Float32, Float16]
metal_dtypes = [Float32, Float16]   # Float64 unsupported on Apple GPU

rows = []
for dt in cpu_dtypes
    @printf "CPU  %-10s …\n" string(dt)
    r = run_bench(G, dt)
    push!(rows, (dtype=string(dt), device="CPU", SVD_s=r.t_svd, SVD_err=r.acc_svd, NS_s=r.t_ns, NS_err=r.acc_ns))
end
for dt in metal_dtypes
    @printf "Metal %-10s …\n" string(dt)
    r = run_bench_metal(G, dt)
    push!(rows, (dtype=string(dt), device="Metal GPU", SVD_s=NaN, SVD_err=NaN, NS_s=r.t_ns, NS_err=r.acc_ns))
end

df = DataFrame(rows)
println(df)

CPU  Float64    …
CPU  Float32    …
CPU  Float16    …
Metal Float32    …
Metal Float16    …
5×6 DataFrame
 Row │ dtype    device     SVD_s       SVD_err        NS_s      NS_err      
     │ String   String     Float64     Float64        Float64   Float64     
─────┼──────────────────────────────────────────────────────────────────────
   1 │ Float64  CPU          1.20857     2.37936e-13  1.66654   1.38556e-14
   2 │ Float32  CPU          0.873208    0.000185945  0.904662  4.93166e-6
   3 │ Float16  CPU          0.85718     0.000188053  0.915951  4.81348e-6
   4 │ Float32  Metal GPU  NaN         NaN            4.60022   6.15314e-5
   5 │ Float16  Metal GPU  NaN         NaN            4.45284   0.0229897


## Q2. Introduction to optimal design

In this exercise, we practice using disciplined convex programming (SDP in particular) to solve optimal design problems.

Consider a linear model
\begin{eqnarray*}
	y_i = \mathbf{x}_i^T \boldsymbol{\beta} + \epsilon_i, \quad i = 1,\ldots, n,
\end{eqnarray*}
where $\epsilon_i$ are independent Gaussian noises with common variance $\sigma^2$. It is well known that the least squares estimate $\hat{\boldsymbol{\beta}}$ is unbiased and has covariance $\sigma^2 (\sum_{i=1}^n \mathbf{x}_i \mathbf{x}_i^T)^{-1}$. 

In **exact optimal design**, given total number of $n$ allowable experiments, we want to choose among a list of $m$ candidate design points $\{\mathbf{x}_1, \ldots, \mathbf{x}_m\}$ such that the covariance matrix is minimized in some sense. In mathematical terms, we want to find an integer vector $\mathbf{n} = (n_1, \ldots, n_m)$ such that $n_i \ge 0$, $\sum_{i=1}^m n_i = n$, and the matrix $\mathbf{V} = \left( \sum_{i=1}^m n_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1}$ is "small."

In **approximate optimal design**,  we want to find a probability vector $\mathbf{p} = (p_1, \ldots, p_m)$ such that $p_i \ge 0$, $\sum_{i=1}^m p_i = 1$, and the matrix $\mathbf{V} = \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1}$ is "small."

Commonly used optimal design criteria include:

- In **$D$-optimal design**, we minimize the determinant of $\mathbf{V}$
\begin{eqnarray*}
	&\text{minimize}& \det \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1} \\
	&\text{subject to}& p_i \ge 0, \sum_{i=1}^m p_i = 1.
\end{eqnarray*}

- In **$E$-optimal design**, we minimize the spectral norm, i.e., the maximum eigenvalue of $\mathbf{V}$
\begin{eqnarray*}
	&\text{minimize}& \lambda_{\text{max}} \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1} \\
	&\text{subject to}& p_i \ge 0, \sum_{i=1}^m p_i = 1.	
\end{eqnarray*}
Statistically we are minimizing the maximum variance of $\sum_{j=1}^p a_j \text{var}(\hat \beta_j)$ over all vectors $\mathbf{a}$ with unit norm.

- In **$A$-optimal design**, we minimize the trace of $\mathbf{V}$
\begin{eqnarray*}
	&\text{minimize}& \text{tr} \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1} \\
	&\text{subject to}& p_i \ge 0, \sum_{i=1}^m p_i = 1.
\end{eqnarray*}
Statistically we are minimizing the total variance $\sum_{j=1}^p \text{var}(\hat \beta_j)$.

### Q2.1 (10 pts) 3x4 factorial design

A drug company asks you to help design a two factor clinical trial, in which treatment A has three levels (A1, A2, and A3) and treatment B has four levels (B1, B2, B3, and B4). Drug company also tells you that the treatment combination A3:B4 has undesirable side effects so we ignore this design point. 

Using dummy coding with A1 and B1 as the baseline levels, find the matrix $C$ with each row a unique design point. Please include the intercept term in the design matrix.

### Q2.2 (30 pts) Find approximate optimal designs

Using semidefinite programming (SDP) software to find the approximate D-, E-, and A-optimal designs for this clinical trial.

Hint: This is what I got (using Hypatia solver), which may or may not be correct.

```
Approximate Optimal Design
┌───────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│ design_pt │   D_opt │   E_opt │   A_opt │ D_opt_n │ E_opt_n │ A_opt_n │
│    String │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │
├───────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│      A1B1 │   0.082 │   0.271 │   0.200 │   8.000 │  27.000 │  20.000 │
│      A2B1 │   0.082 │   0.150 │   0.101 │   8.000 │  15.000 │  10.000 │
│      A3B1 │   0.097 │   0.118 │   0.104 │  10.000 │  12.000 │  10.000 │
│      A1B2 │   0.082 │   0.058 │   0.086 │   8.000 │   6.000 │   9.000 │
│      A2B2 │   0.082 │   0.037 │   0.051 │   8.000 │   4.000 │   5.000 │
│      A3B2 │   0.097 │   0.059 │   0.068 │  10.000 │   6.000 │   7.000 │
│      A1B3 │   0.082 │   0.058 │   0.086 │   8.000 │   6.000 │   9.000 │
│      A2B3 │   0.082 │   0.037 │   0.051 │   8.000 │   3.000 │   5.000 │
│      A3B3 │   0.097 │   0.059 │   0.068 │  10.000 │   6.000 │   7.000 │
│      A1B4 │   0.109 │   0.079 │   0.106 │  11.000 │   8.000 │  10.000 │
│      A2B4 │   0.109 │   0.075 │   0.080 │  11.000 │   7.000 │   8.000 │
│       Obj │   8.987 │  13.000 │  38.925 │   8.988 │  13.028 │  38.946 │
└───────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┘
```

### Q2.3 (30 points) Optimal design with nuisance parameters

Suppose the regression coefficients of linear model $\boldsymbol{\beta}$ is partitioned as $\boldsymbol{\beta} = (\boldsymbol{\beta}_0^T, \boldsymbol{\beta}_1^T)^T$, where $\boldsymbol{\beta}_0$ are nuisance parameters and $\boldsymbol{\beta}_1$ are parameters of primary interest. Given an approximate design $\mathbf{p} = (p_1, \ldots, p_m)$, let the information matrix be partitioned accordingly
$$
\mathbf{I}(\mathbf{p}) = \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T =  \begin{pmatrix}
\mathbf{I}_{00} & \mathbf{I}_{01} \\
\mathbf{I}_{10} & \mathbf{I}_{11}
\end{pmatrix}.
$$
Then the information matrix for $\boldsymbol{\beta}_1$ adjusted for nuisance parameter $\boldsymbol{\beta}_0$ is
$$
\mathbf{I}_{1 \mid 0}(\mathbf{p}) = \mathbf{I}_{11} - \mathbf{I}_{10} \mathbf{I}_{00}^{-1} \mathbf{I}_{01}.
$$

Revisiting the 3x4 factorial design problem in 2.1, suppose the drug company only cares about the estimation of A treatment effects. Find the approximate D-, E-, and A-optimal designs.

Hint: [Schur complement lemma](https://ucla-biostat-216.github.io/2024fall/slides/11-pd/11-pd.html#schur-complement-test-for-positive-definiteness).